In [ ]:
# ViMMKG — tải nốt ảnh + trích đặc trưng (GPU)
# LƯU Ý: Kaggle TỰ GIẢI NÉN tệp .tar.gz khi tạo dataset, nên không có tarball
# để mở — ảnh nằm sẵn dưới dạng thư mục. Dò theo tệp mốc, không đoán đường dẫn.
import os, json, time, sys, shutil

ROOT = "/kaggle/input"
def find_file(marker):
    for dp, dn, fn in os.walk(ROOT):
        if marker in fn: return dp
    return None

def find_images():
    best, n_best = None, 0
    for dp, dn, fn in os.walk(ROOT):
        n = sum(1 for f in fn if f.endswith(".jpg") and not f.startswith("._"))
        if n > n_best: best, n_best = dp, n
    return best, n_best

DS  = find_file("train2id.txt")
EX  = find_file("viwiki_extracts.json")
TI  = find_file("viwiki_titles.json")
P18 = find_file("p18_names.json")
IMG, N_IMG = find_images()

print(f"dataset          : {DS}")
print(f"viwiki_extracts  : {EX}")
print(f"viwiki_titles    : {TI}")
print(f"p18_names        : {P18}")
print(f"ảnh              : {IMG}  ({N_IMG:,} tệp)")

if not all([DS, EX, TI, P18, IMG]):
    print("\nCÂY /kaggle/input:")
    for dp, dn, fn in os.walk(ROOT):
        d = dp.replace(ROOT, "").count("/")
        if d > 3: continue
        print("  "*d, os.path.basename(dp) or ROOT, f"({len(fn)} tệp)", fn[:4])
    raise SystemExit("thiếu dữ liệu đầu vào")

# chép ảnh sang thư mục ghi được (input chỉ đọc), bỏ tệp rác ._ của macOS
W = "/kaggle/working"; os.makedirs(f"{W}/images", exist_ok=True)
t0 = time.time(); n = 0
for f in os.listdir(IMG):
    if f.endswith(".jpg") and not f.startswith("._"):
        shutil.copy(f"{IMG}/{f}", f"{W}/images/{f}"); n += 1
print(f"chép {n:,} ảnh trong {time.time()-t0:.0f}s")

In [ ]:
# entity list theo đúng thứ tự chỉ số
lines = open(f"{DS}/entity2id.txt").read().split("\n")
N = int(lines[0]); ents = [None]*N
for l in lines[1:]:
    if not l.strip(): continue
    q, i = l.rsplit("\t",1) if "\t" in l else l.rsplit(" ",1)
    ents[int(i)] = q
assert all(e is not None for e in ents), "entity2id.txt có lỗ hổng"
print(f"{N:,} thực thể")
names = json.load(open(f"{P18}/p18_names.json"))
have  = {f[:-4] for f in os.listdir(f"{W}/images") if f.endswith(".jpg")}
todo  = [(q, names[q]) for q in ents if q in names and q not in have]
print(f"đã có {len(have):,} ảnh, còn tải {len(todo):,}")

In [ ]:
# tải ảnh còn thiếu — Commons chặn tốc độ (HTTP 429), phải lùi theo cấp số nhân.
# Cấu hình này đã đo trên máy local: 4 luồng + nghỉ 0,5s => fail ~1,6%
import urllib.request, urllib.parse, urllib.error, threading, queue, collections, random
UA = "ViMMKG-research/0.1 (academic; nhantrnh03@gmail.com)"
q_ = queue.Queue()
for x in todo: q_.put(x)
lock=threading.Lock(); st=collections.Counter(); t0=time.time(); TOT=len(todo)

def work():
    while True:
        try: qid, fn = q_.get_nowait()
        except queue.Empty: return
        url = "https://commons.wikimedia.org/wiki/Special:FilePath/"+urllib.parse.quote(fn)+"?width=224"
        ok=False
        for att in range(4):
            try:
                rq = urllib.request.Request(url, headers={"User-Agent": UA})
                with urllib.request.urlopen(rq, timeout=60) as r: data=r.read()
                if len(data) > 500:
                    open(f"{W}/images/{qid}.jpg","wb").write(data); ok=True
                break
            except urllib.error.HTTPError as e:
                if e.code == 429:
                    with lock: st["429"]+=1
                    time.sleep((2**att)*5 + random.random()*3)
                elif e.code in (403,404):
                    with lock: st[f"http{e.code}"]+=1
                    break
                else: time.sleep(3)
            except Exception: time.sleep(3)
        with lock:
            st["ok" if ok else "fail"]+=1
            n=st["ok"]+st["fail"]
            if n % 500 == 0:
                el=time.time()-t0
                print(f"  {n:>6}/{TOT} ok={st['ok']:,} fail={st['fail']} 429={st['429']} "
                      f"{el/60:.1f}ph còn~{(TOT-n)*el/n/60:.0f}ph", flush=True)
        time.sleep(0.5)

if TOT:
    ths=[threading.Thread(target=work,daemon=True) for _ in range(4)]
    [t.start() for t in ths]; [t.join() for t in ths]
print(f"XONG ok={st['ok']:,} fail={st['fail']:,} 429={st['429']} "
      f"404={st.get('http404',0)} trong {(time.time()-t0)/60:.1f}ph")
print(f"tổng ảnh: {len(os.listdir(f'{W}/images')):,}/{N:,}")

In [ ]:
# Kiểm GPU THẬT trước khi nạp mô hình.
#
# Lần chạy 25/09 hỏng ở đây: Kaggle cấp Tesla P100 (sm_60) còn bản torch cài
# sẵn chỉ có kernel từ sm_70, nên CLIP ném AcceleratorError ngay ở lớp
# embedding. torch.cuda.is_available() trả True nên KHÔNG bắt được — phải thử
# chạy thật một phép nhân ma trận.
#
# Khác notebook APKGC: ở đó phần nặng chạy qua subprocess nên cài lại torch
# giữa chừng là đủ. Ở đây CLIP nạp trong chính tiến trình này, nên nếu phải hạ
# torch thì BẮT BUỘC khởi động lại kernel rồi chạy lại từ đầu — ô này dừng hẳn
# thay vì chạy tiếp với torch cũ đã nạp.
import subprocess, sys

import torch

print("torch:", torch.__version__)
DEV = "cpu"
if torch.cuda.is_available():
    _cap = torch.cuda.get_device_capability()
    _sm = f"sm_{_cap[0]}{_cap[1]}"
    print(f"GPU: {torch.cuda.get_device_name(0)}  {_sm}")
    print("torch hỗ trợ:", torch.cuda.get_arch_list())
    try:
        _ = (torch.randn(8, 8, device="cuda") @ torch.randn(8, 8, device="cuda")).sum().item()
        DEV = "cuda"
        print("\nGPU chạy được.")
    except Exception as e:
        print(f"\nGPU KHÔNG chạy được: {type(e).__name__}")
        print(f"torch {torch.__version__} thiếu kernel cho {_sm}.")
        print("\nĐang cài torch 2.4.1+cu121 (có kernel sm_60)...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                        "torchvision", "torchaudio"], check=False)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "--force-reinstall", "torch==2.4.1",
                        "--index-url", "https://download.pytorch.org/whl/cu121"],
                       check=False)
        _probe = ("import torch;a=torch.randn(8,8,device='cuda');"
                  "print('torch',torch.__version__,'| matmul OK',float((a@a).sum()))")
        print(subprocess.run([sys.executable, "-c", _probe],
                             capture_output=True, text=True).stdout.strip())
        raise SystemExit(
            "\n>>> ĐÃ CÀI torch MỚI. Bản cũ vẫn đang nằm trong bộ nhớ tiến trình "
            "này, nên phải Run > Restart & run all rồi chạy lại từ đầu. <<<")
else:
    print("KHÔNG thấy GPU — sẽ chạy CPU, rất chậm.")

print("thiết bị:", DEV)


In [ ]:
# ĐẶC TRƯNG THỊ GIÁC — CLIP ViT-B/32, lấy pooler_output (đã chiếu, 512-d).
# Đã xác minh local: pooler_output == visual_projection(vision_model(x).pooler_output)
# Bỏ qua nếu bản chạy trước đã sinh ra tệp này (gắn output cũ làm input).
# Kaggle giữ /kaggle/working kể cả khi kernel lỗi, nên phần đã tính không mất.
import glob as _g
_prev = _g.glob("/kaggle/input/**/ViMMKG-visual.pth", recursive=True)
if _prev:
    out = torch.load(_prev[0], map_location="cpu")
    print(f"dùng lại từ lần chạy trước: {tuple(out.shape)}")
    raise SystemExit  # dừng ô này, sang ô sau

from transformers import CLIPModel, CLIPImageProcessor
from PIL import Image
m = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").eval().to(DEV)
p = CLIPImageProcessor.from_pretrained("openai/clip-vit-base-patch32")
DIM = m.config.projection_dim
out = torch.zeros(N, DIM)
idx, ims = [], []
n_ok=n_bad=0; t0=time.time(); BS=128

def _img_feat(o):
    """Lấy vector ảnh bất kể phiên bản transformers.

    transformers 4.x: get_image_features trả thẳng Tensor (B, 512).
    transformers 5.x: trả BaseModelOutputWithPooling, vector nằm ở
    pooler_output (đã qua visual_projection — đã kiểm chứng bằng cách so với
    cm.visual_projection(vision_model(x).pooler_output), khớp tuyệt đối).
    """
    if isinstance(o, torch.Tensor): return o
    if hasattr(o, "pooler_output"):  return o.pooler_output
    raise TypeError(f"không rõ kiểu trả về của get_image_features: {type(o)}")

@torch.no_grad()
def flush():
    global idx, ims
    if not ims: return
    x = p(images=ims, return_tensors="pt").to(DEV)
    out[idx] = _img_feat(m.get_image_features(**x)).float().cpu()
    idx, ims = [], []

for i,q in enumerate(ents):
    f=f"{W}/images/{q}.jpg"
    if not os.path.exists(f): continue
    try:
        ims.append(Image.open(f).convert("RGB")); idx.append(i); n_ok+=1
    except Exception:
        n_bad+=1; continue
    if len(ims)>=BS:
        flush()
        if n_ok % 2048 < BS: print(f"  {n_ok:>6}/{N}  {(time.time()-t0)/60:.1f}ph", flush=True)
flush()
torch.save(out, f"{W}/ViMMKG-visual.pth"); print("  đã lưu visual", flush=True)
print(f"ảnh: {n_ok:,} trích, {n_bad} hỏng, {N-n_ok:,} vector 0 -> {tuple(out.shape)}")

In [ ]:
# ĐẶC TRƯNG VĂN BẢN — multilingual-e5-large (1024-d), mean-pooling có mask.
# e5 yêu cầu tiền tố "passage: ".
import glob as _g
_prev = _g.glob("/kaggle/input/**/ViMMKG-textual.pth", recursive=True)
if _prev:
    outt = torch.load(_prev[0], map_location="cpu")
    print(f"dùng lại từ lần chạy trước: {tuple(outt.shape)}")
    raise SystemExit

from transformers import AutoTokenizer, AutoModel
tk = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-large")
mt = AutoModel.from_pretrained("intfloat/multilingual-e5-large").eval().to(DEV)
ex = json.load(open(f"{EX}/viwiki_extracts.json"))
ti = json.load(open(f"{TI}/viwiki_titles.json"))
DT = mt.config.hidden_size
outt = torch.zeros(N, DT)
idx, txt = [], []
n_ok=0; t0=time.time(); BS=32

@torch.no_grad()
def flushT():
    global idx, txt
    if not txt: return
    b = tk(txt, return_tensors="pt", padding=True, truncation=True, max_length=256).to(DEV)
    h = mt(**b).last_hidden_state
    msk = b["attention_mask"].unsqueeze(-1).float()
    outt[idx] = ((h*msk).sum(1)/msk.sum(1)).float().cpu()
    idx, txt = [], []

for i,q in enumerate(ents):
    t = ex.get(q)
    if not t:
        lb = (ti.get(q) or {}).get("label")
        if not lb: continue
        t = lb
    txt.append("passage: "+t[:2000]); idx.append(i); n_ok+=1
    if len(txt)>=BS:
        flushT()
        if n_ok % 2048 < BS: print(f"  {n_ok:>6}/{N}  {(time.time()-t0)/60:.1f}ph", flush=True)
flushT()
torch.save(outt, f"{W}/ViMMKG-textual.pth"); print("  đã lưu textual", flush=True)
print(f"văn bản: {n_ok:,} trích, {N-n_ok:,} vector 0 -> {tuple(outt.shape)}")

In [ ]:
# missing.json ĐÚNG theo dữ liệu thật.
# Lưu ý: tệp missing.json trong kho mã APKGC liệt kê 2 và 1 chỉ số cho MKG-W
# trong khi embedding MKG-W thật có 537 và 851 hàng toàn 0 — nên cách xử lý mô
# thái thiếu ở đây KHÔNG so sánh trực tiếp được với số MKG-W đã công bố.
miss = {}
for mod, t in (("visual", out), ("textual", outt)):
    z = (t.abs().sum(1)==0).nonzero().flatten().tolist()
    miss[f"ViMMKG-{mod}.pth"] = z
    print(f"ViMMKG-{mod}.pth: {len(z):,}/{N:,} hàng toàn 0 ({len(z)/N*100:.1f}%)")
json.dump(miss, open(f"{W}/missing_ViMMKG.json","w"))

# Gộp ảnh thành MỘT tarball rồi xoá thư mục.
# Vì sao: để 12.000 tệp .jpg rời trong /kaggle/working khiến `kaggle kernels
# output` phải phân trang hàng nghìn tệp — không tải nổi log lẫn .pth. Đã vấp
# thật một lần.
import tarfile, shutil
with tarfile.open(f"{W}/images_full.tar.gz", "w:gz") as tar:
    tar.add(f"{W}/images", arcname="images")
n_img = len(os.listdir(f"{W}/images"))
shutil.rmtree(f"{W}/images")
print(f"\nđã nén {n_img:,} ảnh -> images_full.tar.gz, xoá thư mục rời")
print("output:", sorted(os.listdir(W)))